# 📊 EDA — Customer Persistency & Campaign Analysis
**For: Campaign Analyst**  
**Tables:** Customer Persistency · INF_Customer · Inf_Policy

---
> **How to use this notebook:**  
> Replace the sample data loading section with your actual file paths or DB connection.  
> Each section is self-contained — run top to bottom or jump to what you need.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Data
> 💡 Replace these paths with your actual files (CSV, Excel, or DB query).

In [ ]:
# --- Load your files here ---
# persistency = pd.read_csv('customer_persistency.csv')
# customers   = pd.read_csv('inf_customer.csv')
# policies    = pd.read_csv('inf_policy.csv')

# ---- SAMPLE DATA (remove when using real data) ----
np.random.seed(42)
n = 2000

persistency = pd.DataFrame({
    'Owner_Alpha_Id': [f'C{i:05d}' for i in range(n)],
    'Pers_13':        np.random.choice([0, 1], n, p=[0.25, 0.75]),
    'Pers_25':        np.random.choice([0, 1], n, p=[0.35, 0.65]),
    'INFPREM_m13':    np.random.uniform(5000, 80000, n),
    'INFPREM_m25':    np.random.uniform(3000, 75000, n),
    'Written_PREM':   np.random.uniform(8000, 100000, n),
    'ANP':            np.random.uniform(3000, 90000, n),
})

customers = pd.DataFrame({
    'Customer_Alpha_Id': [f'C{i:05d}' for i in range(n)],
    'EC_Tag':            np.random.choice(['Existing', 'New'], n),
    'New_FTM_IND':       np.random.choice([0, 1], n, p=[0.6, 0.4]),
    'Active_Cust_IND':   np.random.choice([0, 1], n, p=[0.2, 0.8]),
    'Policy_Freq':       np.random.randint(1, 6, n),
    'Purchase_Interval': np.random.randint(1, 36, n),
    'AGE_Range':         np.random.choice(['18-25','26-35','36-45','46-55','56+'], n),
    'Owner_Gender':      np.random.choice(['M', 'F'], n),
    'PREMCLUB_IND':      np.random.choice([0, 1], n, p=[0.7, 0.3]),
    'PREMCLUB_TIER':     np.random.choice(['Silver','Gold','Platinum', None], n),
    'OFW_IND':           np.random.choice([0, 1], n, p=[0.85, 0.15]),
    'WORKSITE_IND':      np.random.choice([0, 1], n, p=[0.75, 0.25]),
    'WITH_EMAIL_IND':    np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'WITH_MOBILE_IND':   np.random.choice([0, 1], n, p=[0.2, 0.8]),
    'MARKETING_CONSENT': np.random.choice([0, 1], n, p=[0.4, 0.6]),
    'OWNER_ANNUAL_INCOME': np.random.choice(['<200k','200k-500k','500k-1M','>1M'], n),
    'OWNER_CIVIL_STATUS': np.random.choice(['Single','Married','Widowed'], n),
    'TRANSACTION_CHANNEL': np.random.choice(['Agency','Bancassurance','Worksite','Digital'], n),
    'SOLICIT_BSE':       [f'BSE{np.random.randint(1,50):03d}' for _ in range(n)],
    'BRANCH_NAME':       np.random.choice(['Makati','BGC','Ortigas','QC','Cebu'], n),
})

policies = pd.DataFrame({
    'POLNUM':           [f'POL{i:06d}' for i in range(n)],
    'OWNER_ALPHA_ID':   [f'C{i:05d}' for i in range(n)],
    'POLICY_STATUS':    np.random.choice(['In-Force','Lapsed','Surrendered','Paid-Up'], n, p=[0.65,0.2,0.1,0.05]),
    'EFFECTIVE_DT':     pd.to_datetime(np.random.choice(pd.date_range('2020-01-01','2024-12-31'), n)),
    'PAYMENT_MODE_DESC': np.random.choice(['Monthly','Quarterly','Semi-Annual','Annual'], n),
    'PRODUCT_CATEGORY': np.random.choice(['Traditional','VUL','Health','Group'], n),
    'PLAN_NAME':        np.random.choice(['Plan A','Plan B','Plan C','Plan D'], n),
    'CAMPAIGN_CODE':    np.random.choice(['CAMP01','CAMP02','CAMP03','CAMP04'], n),
    'ANNPREM_PHP':      np.random.uniform(5000, 100000, n),
    'COVANT_PHP':       np.random.uniform(100000, 5000000, n),
})

print('✅ Data loaded.')
print(f'   Persistency: {persistency.shape} | Customers: {customers.shape} | Policies: {policies.shape}')

## 2. Merge Tables

We maintain **two separate analytical tables** to avoid row inflation from a naive 3-way join.

| Table | Grain | Use For |
|-------|-------|---------|
| `df_customer` | One row per customer | Persistency, demographics, segments, ANP, contactability |
| `df_policy` | One row per policy | Product mix, payment mode, coverage, campaign, issuance trends |

> ⚠️ **Why not one big merge?**  
> A customer with 3 policies would appear 3 times, inflating persistency rates, premium sums, and segment counts.

In [ ]:
# ------------------------------------------------------------------
# CUSTOMER-LEVEL TABLE
# Base: Customer Persistency (one row per customer)
# Enriched with: INF_Customer demographics & behavioral attributes
# ------------------------------------------------------------------
df_customer = persistency.merge(
    customers,
    left_on='Owner_Alpha_Id',
    right_on='Customer_Alpha_Id',
    how='left'
)

assert len(df_customer) == len(persistency), \
    f'⚠️ Customer merge inflated rows: {len(persistency)} → {len(df_customer)}. Check for duplicate Customer_Alpha_Id in INF_Customer.'

print(f'✅ df_customer: {df_customer.shape}  (grain: one row per customer)')

# ------------------------------------------------------------------
# POLICY-LEVEL TABLE
# Base: Inf_Policy (one row per policy)
# Enriched with: Customer Persistency (deduped to 1 row per customer)
#               + INF_Customer
#
# WHY DEDUPE: A customer can appear multiple times in Customer Persistency
# (e.g. one row per policy cohort). We keep the latest/most relevant record
# before joining onto policies to avoid row inflation.
# ------------------------------------------------------------------

# Check if persistency has duplicates on the join key
dup_count = persistency['Owner_Alpha_Id'].duplicated().sum()
if dup_count > 0:
    print(f'⚠️  Found {dup_count} duplicate Owner_Alpha_Id rows in Customer Persistency.')
    print('    Deduping: keeping the row with the highest Written_PREM per customer.')
    persistency_deduped = (
        persistency
        .sort_values('Written_PREM', ascending=False)
        .drop_duplicates(subset='Owner_Alpha_Id', keep='first')
    )
    print(f'    Persistency rows: {len(persistency)} → {len(persistency_deduped)}')
else:
    persistency_deduped = persistency
    print('✅  No duplicates found in Customer Persistency.')

# Check if customers has duplicates on the join key
dup_cust = customers['Customer_Alpha_Id'].duplicated().sum()
if dup_cust > 0:
    print(f'⚠️  Found {dup_cust} duplicate Customer_Alpha_Id rows in INF_Customer. Deduping.')
    customers_deduped = customers.drop_duplicates(subset='Customer_Alpha_Id', keep='first')
else:
    customers_deduped = customers

df_policy = policies.merge(
    persistency_deduped,
    left_on='OWNER_ALPHA_ID',
    right_on='Owner_Alpha_Id',
    how='left'
).merge(
    customers_deduped,
    left_on='OWNER_ALPHA_ID',
    right_on='Customer_Alpha_Id',
    how='left'
)

assert len(df_policy) == len(policies), \
    f'⚠️ Policy merge still inflated: {len(policies)} → {len(df_policy)}. Check for duplicate POLNUM in Inf_Policy.'

print(f'✅ df_policy:   {df_policy.shape}  (grain: one row per policy)')
print()
print('Use df_customer for: persistency rates, segments, ANP, contactability, agent/branch performance')
print('Use df_policy   for: product mix, payment mode, campaign attribution, issuance trends')


## 3. Quick Data Health Check

In [ ]:
# Missing values
missing = df_customer.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]

print('=== Columns with missing values ===')
print(missing.apply(lambda x: f'{x:.1%}').to_string())

print('\n=== Data types ===')
print(df.dtypes.value_counts())

In [ ]:
# Customer-level numeric stats
print('--- Customer-level ---')
df_customer[['Written_PREM','INFPREM_m13','INFPREM_m25','ANP','Policy_Freq']].describe().T.round(2)


## 4. Persistency Overview
Key question: **How many customers are retained at 13 and 25 months?**

In [ ]:
pers_13_rate = df_customer['Pers_13'].mean()
pers_25_rate = df_customer['Pers_25'].mean()

print(f'13-Month Persistency Rate : {pers_13_rate:.1%}')
print(f'25-Month Persistency Rate : {pers_25_rate:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label, color in zip(
    axes,
    ['Pers_13', 'Pers_25'],
    ['13-Month Persistency', '25-Month Persistency'],
    ['#4C72B0', '#DD8452']
):
    counts = df_customer[col].value_counts().sort_index()
    ax.bar(['Lapsed (0)', 'Active (1)'], counts.values, color=[color, color], alpha=[0.4, 1.0])
    ax.set_title(label)
    ax.set_ylabel('# of Customers')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 10, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Premium Retention Analysis
Persistency ratios: how much of the written premium is still on the books?

In [ ]:
# Compute retention ratios on customer-level table
df_customer['Prem_Retention_13'] = df_customer['INFPREM_m13'] / df_customer['Written_PREM']
df_customer['Prem_Retention_25'] = df_customer['INFPREM_m25'] / df_customer['Written_PREM']

# Also add to policy-level table for product/campaign breakdowns
df_policy['Prem_Retention_13'] = df_policy['INFPREM_m13'] / df_policy['Written_PREM']
df_policy['Prem_Retention_25'] = df_policy['INFPREM_m25'] / df_policy['Written_PREM']

print('=== Premium Retention Ratios (Customer Level) ===')
print(f'  Avg 13-Month Premium Retention: {df_customer["Prem_Retention_13"].mean():.1%}')
print(f'  Avg 25-Month Premium Retention: {df_customer["Prem_Retention_25"].mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(
    axes,
    ['Prem_Retention_13', 'Prem_Retention_25'],
    ['13-Month Premium Retention Ratio', '25-Month Premium Retention Ratio']
):
    ax.hist(df_customer[col].clip(0, 1.5), bins=30, color='steelblue', edgecolor='white')
    ax.axvline(1.0, color='red', linestyle='--', label='100% retention')
    ax.set_title(title)
    ax.set_xlabel('Retention Ratio')
    ax.legend()

plt.tight_layout()
plt.show()


## 6. Persistency by Key Segments
Break down retention by demographics and customer attributes.

In [ ]:
def plot_persistency_by(col, label, table=None, figsize=(10, 4)):
    """table defaults to df_customer (customer-level). Pass df_policy for product/campaign breakdowns."""
    data = table if table is not None else df_customer
    grp = data.groupby(col)[['Pers_13', 'Pers_25']].mean().sort_values('Pers_13', ascending=False)
    grp.plot(kind='bar', figsize=figsize, color=['#4C72B0','#DD8452'], edgecolor='white')
    plt.title(f'Persistency Rate by {label}')
    plt.ylabel('Rate')
    plt.xlabel(label)
    plt.xticks(rotation=30, ha='right')
    plt.ylim(0, 1)
    plt.legend(['Pers 13', 'Pers 25'])
    plt.tight_layout()
    plt.show()
    return grp.round(3)


In [ ]:
# By Gender
plot_persistency_by('Owner_Gender', 'Gender')

In [ ]:
plot_persistency_by('PRODUCT_CATEGORY', 'Product Category', table=df_policy)


In [ ]:
plot_persistency_by('PAYMENT_MODE_DESC', 'Payment Mode', table=df_policy)


In [ ]:
# By Transaction Channel
plot_persistency_by('TRANSACTION_CHANNEL', 'Channel')

## 7. Campaign Performance
Which campaigns produce the most policies and the best retention?

In [ ]:
campaign_summary = df_policy.groupby('CAMPAIGN_CODE').agg(
    Policy_Count   = ('POLNUM', 'count'),
    Pers_13_Rate   = ('Pers_13', 'mean'),
    Pers_25_Rate   = ('Pers_25', 'mean'),
    Total_Written  = ('Written_PREM', 'sum'),
    Avg_Written    = ('Written_PREM', 'mean'),
    Premium_Ret_13 = ('Prem_Retention_13', 'mean'),
).round(3).sort_values('Pers_13_Rate', ascending=False)

campaign_summary['Total_Written'] = campaign_summary['Total_Written'].map('{:,.0f}'.format)
campaign_summary['Avg_Written']   = campaign_summary['Avg_Written'].map('{:,.0f}'.format)
campaign_summary

In [ ]:
plot_persistency_by('CAMPAIGN_CODE', 'Campaign Code', table=df_policy)


## 8. Customer Segments
New vs Existing, Premier Club, Contactability

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

segment_cols = [
    ('EC_Tag',        'New vs Existing Customer'),
    ('PREMCLUB_IND',  'Premier Club Member'),
    ('New_FTM_IND',   'First-Time Buyer'),
]

for ax, (col, label) in zip(axes, segment_cols):
    grp = df_customer.groupby(col)[['Pers_13','Pers_25']].mean()
    grp.plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452'], edgecolor='white', legend=False)
    ax.set_title(label)
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

axes[0].legend(['Pers 13', 'Pers 25'])
plt.suptitle('Persistency by Customer Segment', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Contactability & Consent
Are contactable customers more persistent?

In [ ]:
contact_cols = ['WITH_EMAIL_IND', 'WITH_MOBILE_IND', 'MARKETING_CONSENT']

contact_df = pd.DataFrame({
    col: df_customer.groupby(col)[['Pers_13','Pers_25']].mean().loc[1]
    for col in contact_cols
}).T

contact_df.index = ['Has Email', 'Has Mobile', 'Marketing Consent']

contact_df.plot(kind='barh', figsize=(9, 4), color=['#4C72B0','#DD8452'], edgecolor='white')
plt.title('Persistency Rate — Contactable Customers (flag = 1)')
plt.xlabel('Rate')
plt.xlim(0, 1)
plt.legend(['Pers 13', 'Pers 25'])
plt.tight_layout()
plt.show()

## 10. Branch & Agent Performance

In [ ]:
branch_perf = df_customer.groupby('BRANCH_NAME').agg(
    Policies     = ('POLNUM', 'count'),
    Pers_13_Rate = ('Pers_13', 'mean'),
    Pers_25_Rate = ('Pers_25', 'mean'),
    Written_PREM = ('Written_PREM', 'sum'),
).sort_values('Pers_13_Rate', ascending=False).round(3)

print('=== Branch Performance ===')
branch_perf

In [ ]:
# Top 15 Agents by Pers 13
agent_perf = df_customer.groupby('SOLICIT_BSE').agg(
    Policies     = ('POLNUM', 'count'),
    Pers_13_Rate = ('Pers_13', 'mean'),
    Pers_25_Rate = ('Pers_25', 'mean'),
).query('Policies >= 5').sort_values('Pers_13_Rate', ascending=False).head(15).round(3)

print('=== Top 15 Agents (min 5 policies) ===')
agent_perf

## 11. Policy Volume Over Time

In [ ]:
df_policy['Issue_YearMonth'] = df_policy['EFFECTIVE_DT'].dt.to_period('M')

monthly = df_policy.groupby('Issue_YearMonth').agg(
    New_Policies = ('POLNUM', 'count'),
    Written_PREM = ('Written_PREM', 'sum')
)

fig, ax1 = plt.subplots(figsize=(13, 4))
ax1.bar(monthly.index.astype(str), monthly['New_Policies'], color='steelblue', alpha=0.6, label='Policies')
ax1.set_ylabel('# of Policies', color='steelblue')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly.index.astype(str), monthly['Written_PREM'], color='orange', linewidth=2, label='Written Prem')
ax2.set_ylabel('Written Premium (PHP)', color='orange')

plt.title('Monthly Policy Issuance & Written Premium')
fig.tight_layout()
plt.show()

## 12. Summary Table — Key Metrics by Product Category

In [ ]:
summary = df_policy.groupby('PRODUCT_CATEGORY').agg(
    Policies         = ('POLNUM', 'count'),
    Pers_13_Rate     = ('Pers_13', 'mean'),
    Pers_25_Rate     = ('Pers_25', 'mean'),
    Avg_Written_PREM = ('Written_PREM', 'mean'),
    Prem_Ret_13      = ('Prem_Retention_13', 'mean'),
    Prem_Ret_25      = ('Prem_Retention_25', 'mean'),
).round(3)

summary.style.background_gradient(subset=['Pers_13_Rate','Pers_25_Rate'], cmap='RdYlGn')


---
## 13. ANP Analysis
**ANP = Annualized New Premium** — measures the value of new business, not just volume.

In [ ]:
print('=== ANP Overview ===')
print(df_customer['ANP'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution
axes[0].hist(df_customer['ANP'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df_customer['ANP'].median(), color='red', linestyle='--', label=f'Median: {df_customer["ANP"].median():,.0f}')
axes[0].set_title('ANP Distribution')
axes[0].set_xlabel('ANP (PHP)')
axes[0].legend()

# ANP vs Written Premium
axes[1].scatter(df_customer['Written_PREM'], df_customer['ANP'], alpha=0.3, color='steelblue', s=10)
axes[1].set_title('ANP vs Written Premium')
axes[1].set_xlabel('Written Premium (PHP)')
axes[1].set_ylabel('ANP (PHP)')

plt.tight_layout()
plt.show()

### 13a. Campaign Quality — ANP per Campaign

In [ ]:
camp_anp = df_policy.groupby('CAMPAIGN_CODE').agg(
    Policy_Count = ('POLNUM', 'count'),
    Total_ANP    = ('ANP', 'sum'),
    Avg_ANP      = ('ANP', 'mean'),
    Pers_13_Rate = ('Pers_13', 'mean'),
    Pers_25_Rate = ('Pers_25', 'mean'),
).round(2).sort_values('Total_ANP', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(camp_anp.index, camp_anp['Total_ANP'], color='steelblue', edgecolor='white')
axes[0].set_title('Total ANP by Campaign')
axes[0].set_ylabel('Total ANP (PHP)')

axes[1].bar(camp_anp.index, camp_anp['Avg_ANP'], color='#DD8452', edgecolor='white')
axes[1].set_title('Avg ANP per Policy by Campaign')
axes[1].set_ylabel('Avg ANP (PHP)')

plt.tight_layout()
plt.show()

camp_anp

### 13b. ANP vs Persistency — Do High-Value Policies Actually Stay?

In [ ]:
# Bucket ANP into tiers
df_customer['ANP_Tier'] = pd.qcut(df_customer['ANP'], q=4, labels=['Low','Mid','High','Top'])

anp_pers = df_customer.groupby('ANP_Tier')[['Pers_13','Pers_25']].mean()

anp_pers.plot(kind='bar', figsize=(8, 4), color=['#4C72B0','#DD8452'], edgecolor='white')
plt.title('Persistency Rate by ANP Tier')
plt.ylabel('Persistency Rate')
plt.xlabel('ANP Tier (quartiles)')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(['Pers 13', 'Pers 25'])
plt.tight_layout()
plt.show()

print(anp_pers.round(3))

### 13c. Agent Performance — ANP Volume vs Quality

In [ ]:
agent_anp = df_customer.groupby('SOLICIT_BSE').agg(
    Policy_Count = ('POLNUM', 'count'),
    Total_ANP    = ('ANP', 'sum'),
    Avg_ANP      = ('ANP', 'mean'),
    Pers_13_Rate = ('Pers_13', 'mean'),
).query('Policy_Count >= 5').round(2)

# Scatter: Volume vs Quality
plt.figure(figsize=(9, 5))
scatter = plt.scatter(
    agent_anp['Total_ANP'],
    agent_anp['Pers_13_Rate'],
    s=agent_anp['Policy_Count'] * 10,
    alpha=0.6,
    color='steelblue',
    edgecolors='white'
)
plt.axhline(agent_anp['Pers_13_Rate'].mean(), color='red', linestyle='--', label='Avg Pers 13')
plt.axvline(agent_anp['Total_ANP'].mean(),    color='orange', linestyle='--', label='Avg ANP')
plt.title('Agent: Total ANP vs Persistency Rate\n(bubble size = # policies)')
plt.xlabel('Total ANP (PHP)')
plt.ylabel('13-Month Persistency Rate')
plt.legend()
plt.tight_layout()
plt.show()

print('Top 10 Agents by Total ANP:')
agent_anp.sort_values('Total_ANP', ascending=False).head(10)

### 13d. ANP by Channel, Product & Customer Segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (col, label, table)
combos = [
    ('TRANSACTION_CHANNEL', 'Channel',           df_customer),
    ('PRODUCT_CATEGORY',    'Product Category',  df_policy),
    ('EC_Tag',              'New vs Existing',   df_customer),
]

for ax, (col, label, tbl) in zip(axes, combos):
    grp = tbl.groupby(col)['ANP'].mean().sort_values(ascending=False)
    ax.bar(grp.index, grp.values, color='steelblue', edgecolor='white')
    ax.set_title(f'Avg ANP by {label}')
    ax.set_ylabel('Avg ANP (PHP)')
    ax.set_xticklabels(grp.index, rotation=20, ha='right')

plt.tight_layout()
plt.show()


### 13e. Customer Value Matrix — ANP × Policy Frequency

In [ ]:
# High ANP + High Policy Freq = most valuable customers
df_customer['ANP_Tier'] = pd.qcut(df_customer['ANP'], q=3, labels=['Low ANP','Mid ANP','High ANP'])
df_customer['Freq_Tier'] = pd.cut(df_customer['Policy_Freq'], bins=[0,1,2,10], labels=['1 Policy','2 Policies','3+ Policies'])

matrix = df_customer.groupby(['Freq_Tier','ANP_Tier'])['Owner_Alpha_Id'].count().unstack()

sns.heatmap(matrix, annot=True, fmt='.0f', cmap='Blues', linewidths=0.5)
plt.title('Customer Value Matrix: Policy Frequency × ANP Tier\n(cell = # of customers)')
plt.tight_layout()
plt.show()

print('\nHigh ANP + 3+ Policies = Premier Club / Upsell targets')
top_value = df_customer[(df_customer['ANP_Tier']=='High ANP') & (df_customer['Freq_Tier']=='3+ Policies')]
print(f'Count: {len(top_value):,} customers')

---
## ✅ EDA Complete

**Key things to check after running:**
- Which campaigns have the highest persistency rate (Section 7)?
- Which channels retain customers best (Section 6)?
- Are contactable customers more retained (Section 9)?
- Which branches / agents are top performers (Section 10)?
- Are premium retention ratios healthy (Section 5)?
- Which campaigns produce the highest ANP — volume vs quality (Section 13a)?
- Do high-ANP policies actually persist (Section 13b)?
- Who are your highest-value customers for upsell (Section 13e)?

> Next steps: filter to specific campaigns, build a lapse prediction model using `Pers_13` / `Pers_25` as the target variable.